# Week 11B — Foundation Segmentation (SAM) and Anomaly Detection

**MCTA 4364 Machine Vision** · Session 2 of Week 11 · 80 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W11_segmentation_anomaly/W11B_sam_anomaly_detection.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W11_segmentation_anomaly/W11B_sam_anomaly_detection.ipynb)

Two problems that break the "collect labels, train a model" recipe:
1. **Labelling masks is slow** (a polygon per object). The **Segment Anything Model (SAM)** turns a click or a box into a mask for objects it has
   never seen — a foundation model for segmentation.
2. **Defects are rare and unpredictable.** You cannot collect 500 examples of a defect that has not happened yet. **Anomaly detection** learns
   what *normal* looks like from good parts only, and flags anything different — including defect types never seen before.

### Learning outcomes
By the end of this session you will be able to:
1. Explain SAM's promptable design (image encoder, prompt encoder, mask decoder) and the SAM family (MobileSAM, SAM 2, SAM 3).
2. Prompt SAM with **boxes** and **points** (positive and negative), measure mask quality, and combine a detector with SAM for **auto-labelling**.
3. Explain the main families of **unsupervised anomaly detection** and implement **PatchCore** (memory bank of pretrained patch features).
4. Evaluate with **AUROC**, anomaly maps and a threshold chosen for a target false-reject rate.
5. Explain why **pose normalisation** and a representative "good" set matter, and where anomaly detection fails (logical anomalies).

### Session plan (80 min)
| Time | Section | Activity |
|---|---|---|
| 0–10 | 1 | Segment Anything: design and family |
| 10–28 | 2–3 | Box and point prompts (interactive); detector + SAM auto-labelling |
| 28–38 | 4–5 | Anomaly detection: families, metrics; golden-template baseline |
| 38–62 | 6–7 | PatchCore: memory bank, coreset, anomaly maps; fixtured vs loose parts |
| 62–72 | 8–9 | Choosing a threshold; MVTec AD and VisA |
| 72–80 | 12 | Quiz and exit ticket |
| Home | 10–11 | Exercises with self-checks, challenge |

**Hardware:** laptop CPU (about 3 minutes). Weights: MobileSAM (40 MB) and a YOLO11n-cls ImageNet backbone (6 MB), downloaded once.

> **Licences.** SAM, SAM 2 and MobileSAM weights are Apache-2.0; the `ultralytics` wrapper is AGPL-3.0 (Meta's `segment-anything` / `sam2` packages
> are permissive alternatives). MVTec AD is CC BY-NC-SA 4.0 (non-commercial, download it yourself — it must not be redistributed); VisA is CC BY 4.0.

## 0. Setup

In [ ]:
# Setup: works in Google Colab, Kaggle, Jupyter/VS Code on your laptop, and CI.
import importlib.util
import os
import subprocess
import sys

REPO_URL = "https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git"
if not os.path.exists("resources/scripts/cvhelpers.py"):
    if os.path.exists("../../resources/scripts/cvhelpers.py"):  # opened from its own folder
        os.chdir("../..")
    else:  # Colab / Kaggle: fetch the course helpers
        if not os.path.isdir("MCTA-4364-Machine-Vision"):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
        os.chdir("MCTA-4364-Machine-Vision")
print("Working directory:", os.getcwd())

# Install only what is missing. PyTorch is deliberately NOT pip-installed here:
# that would replace your laptop's CUDA build with a CPU-only one (see SETUP.md).
needed = {"cv2": "opencv-python", "skimage": "scikit-image", "ipywidgets": "ipywidgets",
          "matplotlib": "matplotlib", "scipy": "scipy", "plotly": "plotly",
          "ultralytics": "ultralytics"}
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
if importlib.util.find_spec("torch") is None:
    raise ImportError("PyTorch is missing. Install it with CUDA support as described in "
                      "SETUP.md, or use Google Colab where it is preinstalled.")
sys.path.append("resources/scripts")

In [ ]:
import time
from pathlib import Path

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from ultralytics import SAM, YOLO

import partsim
from cvhelpers import SMOKE, check, data_dir, interact, pipeline_diagram, quiz, show
from dlhelpers import hw_profile, set_seed

plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 11})
HW = hw_profile()
set_seed(0)
WEIGHTS = data_dir("weights")
sam = SAM(str(WEIGHTS / "mobile_sam.pt"))
scene, s_boxes, s_labels, s_masks = partsim.make_scene(4, return_masks=True)
print(f"conveyor scene with {len(s_boxes)} parts; SAM model: MobileSAM")

## 1. Segment Anything

**SAM** (Kirillov et al., 2023) was trained on **SA-1B**: 11 million images and 1.1 billion masks, labelled with a model-in-the-loop data engine
(Week 9A's flywheel at internet scale). Its design separates the expensive and the cheap parts:

| component | what it does | cost |
|---|---|---|
| **image encoder** (a ViT) | turns the image into an embedding | heavy, **once per image** |
| **prompt encoder** | encodes points (positive / negative), boxes or a coarse mask | tiny |
| **mask decoder** | combines both, outputs **3 candidate masks** + a predicted IoU each (a click is ambiguous: part, object or group?) | milliseconds |

So after one encoding you can prompt interactively in real time. The family keeps growing: **MobileSAM / EfficientSAM** (distilled small encoders
for laptops), **SAM 2** (2024; video, with a memory of previous frames), **SAM 3** (2025; prompts by *concepts* such as "hex nut", i.e.
open-vocabulary segmentation).

In [ ]:
fig, ax = plt.subplots(figsize=(22, 2.6))
pipeline_diagram(["image", "image encoder\n(ViT, once)", "image embedding", "prompt encoder\n(points, box, mask)", "mask decoder\n(fast)",
                  "3 masks +\npredicted IoU"], colors=["#f1f3f4", "#e8f0fe", "#e8f0fe", "#fef7e0", "#e6f4ea", "#f1f3f4"], ax=ax, fontsize=10)
ax.set_title("Figure 1.1 - SAM: encode the image once, then answer many prompts in milliseconds", fontweight="bold")
plt.show()

## 2. Prompting with boxes

A **box** is the least ambiguous prompt: SAM returns the object inside it. Here the boxes come from the ground truth; in practice they come from a
detector (Section 3). We measure each mask's IoU with the true mask.

In [ ]:
def mask_iou(a, b):
    return np.logical_and(a, b).sum() / max(np.logical_or(a, b).sum(), 1)


t0 = time.perf_counter()
r = sam.predict(scene, bboxes=s_boxes.tolist(), verbose=False)[0]
t_box = time.perf_counter() - t0
pred_masks = r.masks.data.cpu().numpy() > 0.5
box_ious = [mask_iou(p, g) for p, g in zip(pred_masks, s_masks)]
vis = scene.copy()
for k, (m, b) in enumerate(zip(pred_masks, s_boxes)):
    col = np.array(plt.cm.tab10(k % 10)[:3]) * 255
    vis[m] = (0.45 * vis[m] + 0.55 * col[::-1]).astype(np.uint8)
    cv2.rectangle(vis, tuple(b[:2].astype(int)), tuple(b[2:].astype(int)), (255, 255, 255), 1)
    cv2.putText(vis, f"IoU {box_ious[k]:.2f}", (int(b[0]), int(b[1]) - 4 if b[1] > 16 else int(b[3]) + 14), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1)
show(vis, titles=[f"box prompts -> {len(pred_masks)} masks, mean IoU with ground truth {np.mean(box_ious):.3f} ({t_box:.1f} s on CPU incl. encoding)"],
     figsize=(11, 8))

## 3. Prompting with points, and auto-labelling

A **point** is ambiguous. Clicking the **centre** of a washer lands in its **hole**, so SAM segments the hole (it is doing exactly what it was
asked). Fixes: click on the material, add **negative points** (label 0) on regions to exclude, or use a box. Move the click below and watch the mask.

In [ ]:
target = int(np.argmax([(b[2] - b[0]) * (b[3] - b[1]) for b in s_boxes]))       # the largest part
bx = s_boxes[target]
cxy = np.array([(bx[0] + bx[2]) / 2, (bx[1] + bx[3]) / 2])


def point_demo(dx=0, dy=0, negative_in_hole=False):
    p = cxy + [dx, dy]
    pts, labels = [p.tolist()], [1]
    if negative_in_hole:
        pts.append(cxy.tolist()); labels.append(0)
    res = sam.predict(scene, points=[pts], labels=[labels], verbose=False)[0]
    m = res.masks.data.cpu().numpy()[0] > 0.5
    vis = scene.copy(); vis[m] = (0.4 * vis[m] + [0, 150, 255]).clip(0, 255).astype(np.uint8)
    for q, lab in zip(pts, labels):
        cv2.circle(vis, tuple(np.round(q).astype(int)), 5, (0, 255, 0) if lab else (0, 0, 255), -1)
    x0, y0, x1, y1 = (bx + [-30, -30, 30, 30]).astype(int).clip(0)
    show(vis[y0:y1, x0:x1], titles=[f"IoU with the true part mask: {mask_iou(m, s_masks[target]):.2f}  (green = positive click, red = negative)"],
         figsize=(6, 6))


interact(point_demo, dx=widgets.IntSlider(value=0, min=-40, max=40), dy=widgets.IntSlider(value=0, min=-40, max=40), negative_in_hole=False)

In [ ]:
res_pts = {"click at the box centre": [], "click on the material": []}
for b, g in zip(s_boxes, s_masks):
    centre = np.array([(b[0] + b[2]) / 2, (b[1] + b[3]) / 2])
    dist = cv2.distanceTransform(g.astype(np.uint8), cv2.DIST_L2, 5)          # the "thickest" point of the part itself
    y, x = np.unravel_index(np.argmax(dist), dist.shape)
    for name, p in [("click at the box centre", centre), ("click on the material", np.array([x, y], float))]:
        m = sam.predict(scene, points=[[p.tolist()]], labels=[[1]], verbose=False)[0].masks.data.cpu().numpy()[0] > 0.5
        res_pts[name].append(mask_iou(m, g))
res_pts["box prompt"] = box_ious
plt.figure(figsize=(12, 3.8))
xs = np.arange(len(s_boxes))
for k, (name, v) in enumerate(res_pts.items()):
    plt.bar(xs + (k - 1) * 0.27, v, 0.27, label=f"{name} (mean {np.mean(v):.2f})")
plt.xticks(xs, [partsim.SCENE_CLASSES[c] for c in s_labels]); plt.ylabel("mask IoU"); plt.ylim(0, 1.05); plt.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.0, 1.0)); plt.grid(alpha=0.3, axis="y")
plt.title("Figure 3.1 - Prompt type matters: a centre click on a part with a hole segments the hole", fontweight="bold"); plt.tight_layout(); plt.show()

**Detector + SAM = auto-labelling.** A detector (Week 9B) gives boxes; SAM turns every box into a mask; a human only **reviews and corrects**. This
is how most segmentation datasets are built today (Week 13A goes further with text prompts). Below: boxes with realistic localisation noise (as from a
detector) are enough for good masks.

In [ ]:
rng = np.random.default_rng(0)
noisy = s_boxes + rng.normal(0, 4, s_boxes.shape)
m_noisy = sam.predict(scene, bboxes=noisy.tolist(), verbose=False)[0].masks.data.cpu().numpy() > 0.5
iou_noisy = [mask_iou(p, g) for p, g in zip(m_noisy, s_masks)]
print(f"mask IoU from exact boxes {np.mean(box_ious):.3f} | from boxes with 4 px noise {np.mean(iou_noisy):.3f}")

## 4. Anomaly detection: learning "normal"

**Setting.** Training data: **good parts only**. Test: good and defective parts, including defect types never seen. Output: an **anomaly score** per
image (reject or accept) and an **anomaly map** per pixel (where).

| family | idea | examples |
|---|---|---|
| reconstruction | an autoencoder / diffusion model reconstructs normal images well, anomalies badly | AE, DRAEM, DiffusionAD |
| **feature embedding (memory bank)** | compare each patch's pretrained-CNN features with the features of good patches | **PatchCore**, PaDiM, SPADE |
| student–teacher | a student network imitates a pretrained teacher only on normal data; disagreement = anomaly | STPM, **EfficientAD** |
| normalising flows | model the density of normal features exactly | CFlow, FastFlow |
| vision–language (zero-shot) | compare patches with text such as "a photo of a damaged part" | WinCLIP, AnomalyCLIP |

**Metrics.** The image-level **AUROC** is the probability that a random defective part scores higher than a random good part (1 = perfect,
0.5 = chance); pixel-level AUROC and PRO measure localisation. In production you then fix a **threshold** (Section 8).

Our data: the flange of Week 7, **fixtured** (always in the same pose, as on a jig) or **loose** (random pose). The "stain" defect is used only
at test time, like a defect that has never happened before.

In [ ]:
N_GOOD = 20 if SMOKE else 60
N_TEST = 6 if SMOKE else 30
DEFECTS = ["missing_hole", "scratch", "chip", "undersize", "stain"]


def parts(defect, n, seed, fixtured=True):
    return [partsim.make_part(seed + i, defect, pose=not fixtured)[0] for i in range(n)]


train_good = parts("ok", N_GOOD, 0)
test_sets = {"ok": parts("ok", N_TEST, 10_000)}
for k, d in enumerate(DEFECTS):
    test_sets[d] = parts(d, N_TEST, 20_000 + 1000 * k)
fig, axes = plt.subplots(1, 6, figsize=(22, 4))
for ax, (name, imgs) in zip(axes, test_sets.items()):
    ax.imshow(imgs[0][..., ::-1]); ax.axis("off"); ax.set_title(name + (" (never seen before)" if name == "stain" else ""))
plt.suptitle("Figure 4.1 - Fixtured test parts: training uses only 'ok' parts", fontweight="bold")
plt.tight_layout(); plt.show()


def auroc_ref(s_good, s_bad):
    s_good, s_bad = np.asarray(s_good), np.asarray(s_bad)
    return float((s_bad[:, None] > s_good[None]).mean() + 0.5 * (s_bad[:, None] == s_good[None]).mean())

## 5. Baseline: golden-template difference

With fixtured parts, the simplest detector is Week 7's **golden template**: the median of the good images; score = the largest (smoothed) absolute
difference. It is fast and explainable, but sensitive to lighting and to normal variation of the surface texture.

In [ ]:
def to_gray(imgs):
    return np.stack([cv2.cvtColor(i, cv2.COLOR_BGR2GRAY).astype(np.float32) for i in imgs])


golden = np.median(to_gray(train_good), 0)


def template_scores(imgs):
    g = to_gray(imgs)
    g = g * (np.median(golden) / np.median(g, axis=(1, 2), keepdims=True))          # simple brightness normalisation
    d = np.abs(g - golden)
    return np.array([cv2.GaussianBlur(x, (0, 0), 3).max() for x in d]), d


tmpl = {name: template_scores(imgs)[0] for name, imgs in test_sets.items()}
auc_tmpl = {d: auroc_ref(tmpl["ok"], tmpl[d]) for d in DEFECTS}
print("golden template AUROC:", {k: round(v, 3) for k, v in auc_tmpl.items()})

## 6. PatchCore

**PatchCore** (Roth et al., 2022) is simple and still one of the strongest methods on MVTec AD:

1. Pass the good images through a CNN **pretrained on ImageNet** (never fine-tuned!) and keep **mid-level** feature maps (edges, textures, small
   shapes — not the ImageNet-specific top layers). Each spatial position is a **patch feature**; a 3 × 3 average adds local context.
2. Put all patch features of all good images in a **memory bank**; shrink it with **coreset subsampling** (greedy k-centre: keep a subset that
   covers the feature space).
3. For a test image, the anomaly score of each patch is the distance to its **nearest neighbour** in the bank; the image score is the maximum.

We use the ImageNet-pretrained backbone of **YOLO11n-cls** (small and fast); the paper uses a WideResNet-50 (also in torchvision).

In [ ]:
backbone = YOLO(str(WEIGHTS / "yolo11n-cls.pt")).model.model.eval()
LAYERS = (4, 6)                                            # two mid-level stages: stride 8 and stride 16


@torch.no_grad()
def patch_features(imgs, size=224):
    x = torch.tensor(np.stack([cv2.resize(i, (size, size))[..., ::-1] for i in imgs]).transpose(0, 3, 1, 2).copy()).float() / 255
    maps, h = {}, x
    for k, layer in enumerate(backbone[:max(LAYERS) + 1]):
        h = layer(h)
        if k in LAYERS:
            maps[k] = F.avg_pool2d(h, 3, 1, 1)             # local context
    ref = maps[LAYERS[0]]
    feats = [ref] + [F.interpolate(maps[k], size=ref.shape[-2:], mode="bilinear", align_corners=False) for k in LAYERS[1:]]
    return torch.cat(feats, 1)                             # [N, C, H, W]


def greedy_coreset_ref(feats, n, seed=0):
    """Greedy k-centre selection: repeatedly add the point farthest from the current selection."""
    feats = torch.as_tensor(feats)
    g = torch.Generator().manual_seed(seed)
    sel = [int(torch.randint(len(feats), (1,), generator=g))]
    dmin = torch.cdist(feats, feats[sel]).squeeze(1)
    for _ in range(n - 1):
        i = int(torch.argmax(dmin)); sel.append(i)
        dmin = torch.minimum(dmin, torch.cdist(feats, feats[i:i + 1]).squeeze(1))
    return torch.tensor(sel)


t0 = time.perf_counter()
Fg = patch_features(train_good)
N_, C_, H_, W_ = Fg.shape
all_patches = Fg.permute(0, 2, 3, 1).reshape(-1, C_)
rand_idx = torch.randperm(len(all_patches), generator=torch.Generator().manual_seed(0))[:20000]   # speed: coreset from a random 20k
core = greedy_coreset_ref(all_patches[rand_idx], 1000 if SMOKE else 4000)
bank = all_patches[rand_idx][core]
print(f"{N_} good images -> {len(all_patches):,} patch features of dimension {C_} -> coreset memory bank of {len(bank):,} "
      f"({time.perf_counter() - t0:.1f} s)")


@torch.no_grad()
def patchcore(imgs, bank=bank):
    f = patch_features(imgs)
    n = f.shape[0]
    q = f.permute(0, 2, 3, 1).reshape(n, -1, C_)
    d = torch.stack([torch.cdist(qi, bank).min(1).values for qi in q]).reshape(n, H_, W_)
    return d.amax((1, 2)).numpy(), d.numpy()


pc = {name: patchcore(imgs) for name, imgs in test_sets.items()}
auc_pc = {d: auroc_ref(pc["ok"][0], pc[d][0]) for d in DEFECTS}
print("PatchCore AUROC:", {k: round(v, 3) for k, v in auc_pc.items()})

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(22, 7.5))
vmax = np.percentile(np.concatenate([pc[d][1].ravel() for d in pc]), 99.5)
for k, name in enumerate(test_sets):
    i = int(np.argmax(pc[name][0])) if name != "ok" else 0
    img = test_sets[name][i]
    amap = cv2.resize(pc[name][1][i], img.shape[1::-1], interpolation=cv2.INTER_CUBIC)
    axes[0, k].imshow(img[..., ::-1]); axes[0, k].set_title(f"{name}: score {pc[name][0][i]:.2f}")
    axes[1, k].imshow(img[..., ::-1]); axes[1, k].imshow(amap, cmap="jet", alpha=0.5, vmin=0, vmax=vmax)
    axes[1, k].set_title("anomaly map")
for ax in axes.ravel():
    ax.axis("off")
plt.suptitle("Figure 6.1 - PatchCore anomaly maps (same colour scale): the defect lights up, even the never-seen stain", fontweight="bold")
plt.tight_layout(); plt.show()

## 7. Fixtured vs loose parts, and what anomaly detection cannot see

PatchCore compares patches **without knowing where they are**. A missing bolt hole is just "more normal metal", which exists elsewhere on a good part:
the patch looks normal. With a **fixture** (or the alignment of Week 7), each position always sees the same structure, and the model effectively
learns *where* holes belong. Below: the same method trained and tested on **loose** parts (random rotation and position).

In [ ]:
train_loose = parts("ok", N_GOOD, 0, fixtured=False)
F_loose = patch_features(train_loose).permute(0, 2, 3, 1).reshape(-1, C_)
idx_l = torch.randperm(len(F_loose), generator=torch.Generator().manual_seed(0))[:20000]
bank_loose = F_loose[idx_l][greedy_coreset_ref(F_loose[idx_l], 1000 if SMOKE else 4000)]
loose_sets = {"ok": parts("ok", N_TEST, 10_000, fixtured=False)}
for k, d in enumerate(DEFECTS):
    loose_sets[d] = parts(d, N_TEST, 20_000 + 1000 * k, fixtured=False)
pl = {name: patchcore(imgs, bank_loose)[0] for name, imgs in loose_sets.items()}
auc_loose = {d: auroc_ref(pl["ok"], pl[d]) for d in DEFECTS}
fig, ax = plt.subplots(figsize=(14, 4.2))
xs = np.arange(len(DEFECTS))
for k, (name, res, col) in enumerate([("golden template (fixtured)", auc_tmpl, "#9aa0a6"), ("PatchCore (fixtured)", auc_pc, "#188038"),
                                      ("PatchCore (loose parts)", auc_loose, "#f29900")]):
    ax.bar(xs + (k - 1) * 0.27, [res[d] for d in DEFECTS], 0.27, label=name, color=col)
ax.axhline(0.5, color="k", ls=":", lw=1); ax.set_xticks(xs); ax.set_xticklabels(DEFECTS); ax.set_ylim(0.4, 1.02); ax.set_ylabel("image AUROC")
ax.legend(loc="lower left"); ax.grid(alpha=0.3, axis="y")
ax.set_title("Figure 7.1 - Local texture defects are easy; 'logical' defects (missing hole, wrong size) need a fixed pose", fontweight="bold")
plt.tight_layout(); plt.show()
for d in DEFECTS:
    print(f"{d:<13} template {auc_tmpl[d]:.3f} | PatchCore fixtured {auc_pc[d]:.3f} | PatchCore loose {auc_loose[d]:.3f}")

## 8. Choosing a threshold

AUROC summarises all thresholds; a production line needs **one**. A common rule: accept a target **false-reject rate** on good parts (e.g. 1%),
set the threshold at the corresponding percentile of the **good** scores (from a separate validation set of good parts), and report the resulting
**detection rate** per defect type. Scores of good parts are cheap to collect; defect scores are not.

In [ ]:
val_good_scores = patchcore(parts("ok", N_TEST, 50_000))[0]


def threshold_demo(false_reject_pct=1.0):
    thr = np.percentile(val_good_scores, 100 - false_reject_pct)
    fig, axes = plt.subplots(1, 2, figsize=(17, 4.2))
    bins = np.linspace(min(pc[k][0].min() for k in pc), max(pc[k][0].max() for k in pc), 40)
    axes[0].hist(pc["ok"][0], bins, alpha=0.7, label="good (test)", color="#188038")
    for d, col in zip(DEFECTS, ["#d93025", "#9334e6", "#f29900", "#1a73e8", "#795548"]):
        axes[0].hist(pc[d][0], bins, alpha=0.45, label=d, color=col)
    axes[0].axvline(thr, color="k", lw=2, label=f"threshold ({false_reject_pct:.1f}% false rejects on validation)")
    axes[0].set_xlabel("image anomaly score"); axes[0].legend(fontsize=7)
    rates = {d: (pc[d][0] > thr).mean() for d in DEFECTS}
    axes[1].bar(list(rates), list(rates.values()), color="#d93025"); axes[1].set_ylim(0, 1.05)
    axes[1].set_title(f"detection rate per defect; good parts rejected on test: {(pc['ok'][0] > thr).mean():.0%}"); axes[1].grid(alpha=0.3, axis="y")
    plt.suptitle("Figure 8.1 - Picking an operating point from good parts only", fontweight="bold"); plt.tight_layout(); plt.show()


interact(threshold_demo, false_reject_pct=widgets.FloatSlider(value=1.0, min=0.0, max=20.0, step=0.5))

## 9. Real data: MVTec AD and VisA

**MVTec AD** (Bergmann et al., 2019) is the standard benchmark: 15 categories (bottle, cable, screw, carpet, ...), 3,629 good training images and
1,725 test images with pixel-precise defect masks. **VisA** (Zou et al., 2022; Amazon) adds 12 harder categories with multiple objects and complex
structures. **MVTec LOCO** focuses on *logical* anomalies (wrong count, wrong position), exactly what Section 7 showed to be hard.

The MVTec licence (CC BY-NC-SA 4.0) allows non-commercial use but **not redistribution**, so the course does not download it for you. Download a
category (e.g. `screw`, 180 MB) from the MVTec website, unpack it to `data/mvtec_ad/screw/`, and run the next cell. VisA (CC BY 4.0) works the same way.

In [ ]:
MVTEC = Path("data/mvtec_ad/screw")
if MVTEC.exists() and not SMOKE:
    load = lambda paths: [cv2.imread(str(p)) for p in paths]
    good_train = load(sorted((MVTEC / "train" / "good").glob("*.png"))[:100])
    Fm = patch_features(good_train).permute(0, 2, 3, 1).reshape(-1, C_)
    idx_m = torch.randperm(len(Fm))[:20000]
    bank_m = Fm[idx_m][greedy_coreset_ref(Fm[idx_m], 4000)]
    scores = {d.name: patchcore(load(sorted(d.glob("*.png"))), bank_m)[0] for d in sorted((MVTEC / "test").iterdir())}
    for d in scores:
        if d != "good":
            print(f"{d:<16} AUROC {auroc_ref(scores['good'], scores[d]):.3f}")
else:
    print("MVTec AD not found at data/mvtec_ad/screw - skipped (see the instructions above).")

## 10. Exercises (with self-checks)
Replace each `raise NotImplementedError`, then run the check below it.

### E1. AUROC
Return the image-level AUROC: the fraction of (defective, good) pairs where the defective score is higher, counting ties as ½. Do not call `auroc_ref`.

In [ ]:
def auroc(scores_good, scores_bad):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e1():
    assert auroc([0.1, 0.2], [0.3, 0.4]) == 1.0, "perfect separation -> 1"
    assert auroc([0.3, 0.4], [0.1, 0.2]) == 0.0, "inverted -> 0"
    assert abs(auroc([0.1, 0.5], [0.5, 0.2]) - 0.625) < 1e-9, "ties count as 1/2: expected 0.625"
    assert abs(auroc(pc["ok"][0], pc["scratch"][0]) - auc_pc["scratch"]) < 1e-9, "differs from Section 6"


check("E1 auroc", _test_e1)

### E2. Nearest-neighbour anomaly map
Given patch features `f` of one image `[C, H, W]` and a memory bank `[M, C]`, return an `[H, W]` NumPy array with, for every position, the Euclidean distance to its **nearest** bank feature.

In [ ]:
def nn_anomaly_map(f, bank):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e2():
    bank_t = torch.tensor([[0.0, 0.0], [3.0, 4.0]])
    f = torch.zeros(2, 1, 2); f[:, 0, 1] = torch.tensor([3.0, 4.0 + 1.0])
    a = np.asarray(nn_anomaly_map(f, bank_t), float)
    assert a.shape == (1, 2) and np.allclose(a, [[0.0, 1.0]]), f"expected [[0, 1]], got {a}"
    img = test_sets["scratch"][0]
    ref = patchcore([img])[1][0]
    got = np.asarray(nn_anomaly_map(patch_features([img])[0], bank), float)
    assert np.allclose(got, ref, atol=1e-4), "differs from patchcore() on a test image"


check("E2 nn_anomaly_map", _test_e2)

### E3. Greedy coreset
Implement `greedy_coreset(feats, n, seed=0)` exactly like Section 6: start from one random index (`torch.randint` with a generator seeded by `seed`), then repeatedly add the point with the largest distance to its nearest selected point. Return a tensor of `n` indices.

In [ ]:
def greedy_coreset(feats, n, seed=0):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e3():
    pts = torch.tensor([[0.0, 0.0], [0.1, 0.0], [10.0, 0.0], [0.0, 10.0], [10.1, 0.0]])    # clusters {0, 1}, {2, 4}, {3}
    sel = set(int(i) for i in np.asarray(greedy_coreset(pts, 3, seed=0)))
    covered = [bool(sel & {0, 1}), bool(sel & {2, 4}), 3 in sel]
    assert len(sel) == 3 and all(covered), f"a 3-point coreset must take one point from each cluster, got {sel}"
    x = torch.randn(300, 8, generator=torch.Generator().manual_seed(1))
    assert np.array_equal(np.asarray(greedy_coreset(x, 20, 3)), greedy_coreset_ref(x, 20, 3).numpy()), "differs from the reference"


check("E3 greedy_coreset", _test_e3)

### E4. Loose parts, fixed by alignment (no self-check)
Use Week 7's `estimate_pose` / `align` functions (copy them here) to align the **loose** parts before PatchCore. How close do you get to the
fixtured AUROC for `missing_hole` and `undersize`? What does this tell you about mechanical fixtures vs software alignment?

In [ ]:
# E4: your code here

## 11. Challenge: SAM for defect measurement
Threshold the PatchCore anomaly map of a scratched or chipped part, take the peak as a **point prompt** for SAM, and compare the SAM mask of the
defect with the thresholded map. Report the defect area in mm² (0.2 mm/px). Which gives the more reliable measurement, and when would SAM segment
the wrong thing?

In [ ]:
# Your challenge code here

## 12. Quick check (auto-marked)

In [ ]:
quiz([
    {
        "q": 'Why can SAM answer many prompts on the same image quickly?',
        "options": [
            'It uses a small image',
            'The heavy image encoder runs once; the prompt decoder is tiny',
            'It skips the image',
            'It uses a GPU only',
        ],
        "answer": '9d871e63ef',
        "explain": 'Encode once, prompt many times.',
    },
    {
        "q": 'A positive click at the centre of a washer returns the hole. What is the best fix?',
        "options": [
            'Retrain SAM',
            'Use a box prompt or add a negative point in the hole',
            'Increase the image size',
            'Use a lower threshold',
        ],
        "answer": 'f78406bc67',
        "explain": 'Clicks are ambiguous; boxes and negative points disambiguate.',
    },
    {
        "q": 'What training data does unsupervised anomaly detection use?',
        "options": [
            'Many labelled defects',
            'Only good (normal) samples',
            'Only defects',
            'Random internet images',
        ],
        "answer": 'c872283c2d',
        "explain": 'Anomalies are whatever differs from normal.',
    },
    {
        "q": 'In PatchCore, what is the anomaly score of a patch?',
        "options": [
            'Its reconstruction error',
            'The distance to its nearest neighbour in the memory bank of good patches',
            'Its class probability',
            'Its brightness',
        ],
        "answer": 'c380f60192',
        "explain": 'Image score = the maximum over patches.',
    },
    {
        "q": 'Why does PatchCore use mid-level (not top-level) features of an ImageNet CNN?',
        "options": [
            'They are faster',
            'They describe local structure and texture, not ImageNet categories',
            'Top layers do not exist',
            'To save memory only',
        ],
        "answer": 'f81bc78d93',
        "explain": 'Top layers are specialised to ImageNet classes.',
    },
    {
        "q": 'Image AUROC = 0.5 means...',
        "options": [
            'perfect detection',
            'no better than chance',
            'half the defects are found at every threshold',
            'the threshold is 0.5',
        ],
        "answer": 'b4c6289a35',
        "explain": 'AUROC is the probability that a defect scores above a good part.',
    },
    {
        "q": 'A missing hole is hard for PatchCore on loose parts because...',
        "options": [
            'holes are dark',
            'each patch still looks like normal metal seen elsewhere on good parts',
            'SAM is needed',
            'the image is too small',
        ],
        "answer": '1ce4196b90',
        "explain": "It is a 'logical' anomaly: needs position (fixture/alignment).",
    },
], title='Week 11B quiz')

## 13. Exit ticket (reflection)
1. Your line produces 1 defect in 10,000 parts. Why is anomaly detection attractive, and how would you validate it without many defect images?
2. Which prompt type would you use to auto-label your project data with SAM, and how would you check the masks?
3. Name one defect type that PatchCore would miss and a way to catch it.

## 14. What's new (2025–26)

> - **SAM 2 and SAM 3**: video segmentation with memory, and concept prompts ("all hex nuts") that merge detection and segmentation; efficient variants
>   (EfficientTAM, EdgeSAM, SAM2.1-tiny) run on laptops and phones.
> - **Anomaly detection**: EfficientAD and PatchCore remain strong baselines; foundation-model methods (AnomalyCLIP, AnomalyGPT, MuSc, UniAD) work
>   zero-/few-shot or with one model for many products; **Anomalib 2** packages them with export to OpenVINO/ONNX.
> - **Logical anomalies** (MVTec LOCO) and **3D/multimodal** inspection (MVTec 3D-AD, Real-IAD) are the active frontiers.

## 15. References and further exploration
- Kirillov, A. et al. (2023). *Segment anything.* ICCV. [arXiv:2304.02643](https://arxiv.org/abs/2304.02643) · [segment-anything.com](https://segment-anything.com/)
- Ravi, N. et al. (2024). *SAM 2: Segment anything in images and videos.* [arXiv:2408.00714](https://arxiv.org/abs/2408.00714)
- Zhang, C. et al. (2023). *Faster segment anything: Towards lightweight SAM for mobile applications* (MobileSAM). [arXiv:2306.14289](https://arxiv.org/abs/2306.14289)
- Roth, K. et al. (2022). *Towards total recall in industrial anomaly detection* (PatchCore). CVPR. [arXiv:2106.08265](https://arxiv.org/abs/2106.08265)
- Defard, T. et al. (2021). *PaDiM: a patch distribution modeling framework for anomaly detection and localization.* [arXiv:2011.08785](https://arxiv.org/abs/2011.08785)
- Batzner, K., Heckler, L. & König, R. (2024). *EfficientAD: Accurate visual anomaly detection at millisecond-level latencies.* WACV. [arXiv:2303.14535](https://arxiv.org/abs/2303.14535)
- Bergmann, P. et al. (2019). *MVTec AD — a comprehensive real-world dataset for unsupervised anomaly detection.* CVPR. [doi:10.1109/CVPR.2019.00982](https://doi.org/10.1109/CVPR.2019.00982)
- Bergmann, P. et al. (2022). *Beyond dents and scratches: logical constraints in unsupervised anomaly detection* (MVTec LOCO). IJCV 130. [doi:10.1007/s11263-022-01578-9](https://doi.org/10.1007/s11263-022-01578-9)
- Zou, Y. et al. (2022). *SPot-the-Difference self-supervised pre-training for anomaly detection and segmentation* (VisA). ECCV. [arXiv:2207.14315](https://arxiv.org/abs/2207.14315)
- Jeong, J. et al. (2023). *WinCLIP: Zero-/few-shot anomaly classification and segmentation.* CVPR. [arXiv:2303.14814](https://arxiv.org/abs/2303.14814)
- Akcay, S. et al. (2022). *Anomalib.* [arXiv:2202.08341](https://arxiv.org/abs/2202.08341) · [github.com/open-edge-platform/anomalib](https://github.com/open-edge-platform/anomalib)

**Data:** parts and scenes generated by `partsim.py`; MobileSAM (Apache-2.0) and YOLO11n-cls (AGPL-3.0) weights via Ultralytics.

## 16. Key takeaways
- **SAM**: encode once, prompt with boxes / points / masks; boxes are the most reliable prompt; detector + SAM = fast mask labelling.
- **Anomaly detection** learns from good parts only and finds defects never seen before.
- **PatchCore** = memory bank of pretrained mid-level patch features + nearest-neighbour distance; **coreset** keeps it small.
- Evaluate with **AUROC** and choose a **threshold** from good parts for a target false-reject rate.
- Local methods miss **logical** anomalies unless the pose is fixed (fixture or alignment).

**Next week (12A):** from 2D to 3D — stereo vision, depth and point clouds.